In [ ]:
%%capture
!pip install --proxy=192.168.2.10:8080 --upgrade snowflake-connector-python[pandas]
!pip install --proxy=192.168.2.10:8080 --upgrade keyring seaborn statsmodels
!pip install --proxy=192.168.2.10:8080 pmdarima
!pip install --proxy=192.168.2.10:8080 tbats
!pip install --proxy=192.168.2.10:8080 hyperopt

In [ ]:
import os
import json
import warnings
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import snowflake.connector
from sklearn.metrics import mean_absolute_percentage_error

# ── Statistical model imports ─────────────────────────────────────────────
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.forecasting.theta import ThetaModel
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import pmdarima as pm
from tbats import TBATS as TBATSEstimator

# ── Hyperopt ──────────────────────────────────────────────────────────────
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval

warnings.filterwarnings('ignore')

# Setting Up the Connector

In [ ]:
with open('./credentials.json', 'r') as config_file:
    config = json.load(config_file)

snowflake_creds = config['snowflake_credentials']

In [ ]:
ctx = snowflake.connector.connect(**snowflake_creds)

In [ ]:
cur = ctx.cursor()

# Load Datasets

In [ ]:
cur.execute("""
SELECT * FROM IRONONETECH_DB.PUBLIC.V_FORECAST_CURVE_CONSOLIDATED;
""")
df = cur.fetch_pandas_all()
df.head()

# Data Preparation

In [ ]:
portfolio       = 'Buckeye'
sub_portfolio   = 'PP'
last_input_date = '2024-12-31'
TARGET_METRIC   = 'Beginning Outstanding Loans'

# Statistical benchmarks for reference
BASELINE_PROPHET_MAPE = 6.54
STATISTICAL_TARGET    = 3.21

In [ ]:
portfolio_df = df[df['PORTFOLIO'] == portfolio]
portfolio_df = portfolio_df[portfolio_df['SUB_PORTFOLIO'] == sub_portfolio]
portfolio_df = portfolio_df[portfolio_df['FORECAST_TYPE'] == 'Actual']

metrics = portfolio_df['METRIC'].unique()
print('Available metrics:', metrics)

In [ ]:
dff = portfolio_df.groupby(['DATE', 'METRIC']).sum().reset_index()
dff

In [ ]:
data_df = dff.pivot(
    index='DATE',
    columns='METRIC',
    values='METRIC_VALUE'
).sort_index()

In [ ]:
train_df = data_df[data_df.index <= datetime.date.fromisoformat(last_input_date)]
test_df  = data_df[data_df.index >  datetime.date.fromisoformat(last_input_date)]

print(f"Train : {len(train_df)} monthly points  ({train_df.index[0]} → {train_df.index[-1]})")
print(f"Test  : {len(test_df)}  monthly points  ({test_df.index[0]}  → {test_df.index[-1]})")
train_df.head()

# Utility Functions

In [ ]:
def plot_results(pred_df, metric):
    plt.figure(figsize=(20, 6))
    plt.title(metric)
    sns.lineplot(data=pred_df, x='DATE', y='METRIC_VALUE',
                 hue='FORECAST_TYPE', errorbar=None)
    plt.xlabel('DATE')
    plt.ylabel('METRIC_VALUE')
    plt.xticks(pred_df['DATE'].unique(), rotation=90)


def add_reforecast_data(pred_df, main_df, portfolio, metric, forecast_types):
    fdf = main_df[
        (main_df['PORTFOLIO'] == portfolio) &
        (main_df['DATE'].astype(str).isin(pred_df['DATE'].astype(str))) &
        (main_df['METRIC'] == metric) &
        (main_df['FORECAST_TYPE'].isin(forecast_types))
    ]
    fdf = fdf[pred_df.columns].copy()
    fdf['DATE'] = fdf['DATE'].apply(
        lambda x: datetime.date.fromisoformat(str(x)))
    return pd.concat([pred_df, fdf], axis=0)


def evaluate(df_check, start, end, metric,
             reforecast_cols=None, original_col='Original'):
    if reforecast_cols is None:
        reforecast_cols = ['2025 9+3']
    df_check = df_check[df_check['METRIC'] == metric]
    df_check = df_check[
        df_check['FORECAST_TYPE'].isin([original_col, 'Actual'] + reforecast_cols)]
    start_date = datetime.date.fromisoformat(start)
    end_date   = datetime.date.fromisoformat(end)
    df_check = df_check[
        (df_check['DATE'] >= start_date) & (df_check['DATE'] <= end_date)]
    y_true = (df_check[df_check['FORECAST_TYPE'] == 'Actual']
              .sort_values('DATE').set_index('DATE')['METRIC_VALUE'])
    result_df = pd.DataFrame({'Actual': y_true.values}, index=y_true.index)
    for col in [original_col] + reforecast_cols:
        y_pred = (df_check[df_check['FORECAST_TYPE'] == col]
                  .sort_values('DATE').set_index('DATE')['METRIC_VALUE'])
        y_pred.name = col
        result_df = result_df.merge(y_pred, how='left',
                                    left_index=True, right_index=True)
    return result_df


def ts_cv_splits(series, n_splits=2, val_size=3, min_train=7):
    """Walk-forward CV splits — no data leakage."""
    n = len(series)
    for fold in range(n_splits):
        val_end   = n - (n_splits - 1 - fold) * val_size
        val_start = val_end - val_size
        if val_start < min_train:
            continue
        yield list(range(val_start)), list(range(val_start, val_end))


def to_long(test_dates, actual_values, pred_values, model_label):
    """Convert test actuals + predictions to the long format used by plot_results."""
    rows = []
    for d, a, p in zip(test_dates, actual_values, pred_values):
        rows.append({'DATE': d, 'FORECAST_TYPE': 'Actual',     'METRIC_VALUE': a})
        rows.append({'DATE': d, 'FORECAST_TYPE': model_label,  'METRIC_VALUE': p})
    return pd.DataFrame(rows)


print("Utility functions loaded.")

# Model 1 — Holt-Winters Exponential Smoothing

**Why:** Handles trend + seasonality with multiplicative error spec for loan balances  
**Seasonal period:** `m=3` (quarterly cycle on monthly data — safe with only ~19 training points)  
**Tuned params:** `trend_type`, `seasonal_type`, `damped_trend`, `alpha`, `beta`, `gamma`, `phi`

In [ ]:
SEASONAL_PERIOD = 3   # quarterly on monthly data (m=12 needs 2+ yrs; too large for 19 pts)

hw_space = {
    'trend'        : hp.choice('hw_trend',    ['add', 'mul']),
    'seasonal'     : hp.choice('hw_seasonal', ['add', 'mul']),
    'damped_trend' : hp.choice('hw_damped',   [True, False]),
    'alpha'        : hp.uniform('hw_alpha',   0.05, 0.99),
    'beta'         : hp.uniform('hw_beta',    0.001, 0.50),
    'gamma'        : hp.uniform('hw_gamma',   0.001, 0.50),
    'phi'          : hp.uniform('hw_phi',     0.80, 0.999),
}


def hw_cv_objective(params):
    series     = train_df[TARGET_METRIC].dropna()
    fold_mapes = []

    for tr_idx, val_idx in ts_cv_splits(series):
        tr  = series.iloc[tr_idx].values
        val = series.iloc[val_idx].values

        # multiplicative seasonal/trend requires strictly positive data
        if params['seasonal'] == 'mul' and np.any(tr <= 0):
            fold_mapes.append(200)
            continue
        try:
            model = ExponentialSmoothing(
                tr,
                trend            = params['trend'],
                seasonal         = params['seasonal'],
                seasonal_periods = SEASONAL_PERIOD,
                damped_trend     = params['damped_trend'],
            )
            fit = model.fit(
                smoothing_level          = params['alpha'],
                smoothing_trend          = params['beta'],
                smoothing_seasonal       = params['gamma'],
                damping_trend            = params['phi'],
                optimized                = False,
            )
            fc   = fit.forecast(len(val_idx))
            fc   = np.maximum(fc, 0)
            mape = mean_absolute_percentage_error(val, fc) * 100
            fold_mapes.append(mape)
        except Exception:
            fold_mapes.append(200)

    return {'loss': np.mean(fold_mapes), 'status': STATUS_OK}


print("Tuning Holt-Winters...")
hw_trials = Trials()
hw_best_raw = fmin(
    fn        = hw_cv_objective,
    space     = hw_space,
    algo      = tpe.suggest,
    max_evals = 100,
    trials    = hw_trials,
    rstate    = np.random.default_rng(42),
    verbose   = False,
)
best_hw_params = space_eval(hw_space, hw_best_raw)

print(f"  Best CV MAPE : {min(hw_trials.losses()):.2f}%")
print(f"  Target MAPE : {STATISTICAL_TARGET}%")
print("  Best params  :", {k: (round(v, 4) if isinstance(v, float) else v)
                           for k, v in best_hw_params.items()})

# Model 2 — ETS (Error/Trend/Seasonal with AIC selection)

**Why:** Most principled smoothing model — picks error/trend/seasonal type by AIC.  
Multiplicative error captures the scaling volatility characteristic of loan balance series.  
**Tuned params:** `error`, `trend`, `seasonal`, `damped_trend` combination best by CV MAPE

In [ ]:
ets_space = {
    'error'        : hp.choice('ets_error',    ['add', 'mul']),
    'trend'        : hp.choice('ets_trend',    ['add', None]),
    'seasonal'     : hp.choice('ets_seasonal', ['add', 'mul', None]),
    'damped_trend' : hp.choice('ets_damped',   [True, False]),
}


def ets_cv_objective(params):
    series     = train_df[TARGET_METRIC].dropna()
    fold_mapes = []

    for tr_idx, val_idx in ts_cv_splits(series):
        tr  = series.iloc[tr_idx].values
        val = series.iloc[val_idx].values

        # multiplicative spec requires all-positive data
        if (params['error'] == 'mul' or params['seasonal'] == 'mul') \
                and np.any(tr <= 0):
            fold_mapes.append(200)
            continue
        # damped_trend only valid when trend is not None
        damped = params['damped_trend'] if params['trend'] is not None else False
        try:
            model = ETSModel(
                tr,
                error            = params['error'],
                trend            = params['trend'],
                seasonal         = params['seasonal'],
                seasonal_periods = SEASONAL_PERIOD if params['seasonal'] else None,
                damped_trend     = damped,
            )
            fit  = model.fit(disp=False)
            fc   = fit.forecast(len(val_idx))
            fc   = np.maximum(fc, 0)
            mape = mean_absolute_percentage_error(val, fc) * 100
            fold_mapes.append(mape)
        except Exception:
            fold_mapes.append(200)

    return {'loss': np.mean(fold_mapes), 'status': STATUS_OK}


print("Tuning ETS...")
ets_trials = Trials()
ets_best_raw = fmin(
    fn        = ets_cv_objective,
    space     = ets_space,
    algo      = tpe.suggest,
    max_evals = 80,
    trials    = ets_trials,
    rstate    = np.random.default_rng(42),
    verbose   = False,
)
best_ets_params = space_eval(ets_space, ets_best_raw)

print(f"  Best CV MAPE : {min(ets_trials.losses()):.2f}%")
print(f"  Target MAPE : {STATISTICAL_TARGET}%")
print("  Best params  :", best_ets_params)

# Model 3 — SARIMAX (auto_arima + exogenous regressors)

**Why:** Best volatility capture — ARMA residual structure allows the model to learn  
autocorrelated noise patterns that smoothing models ignore.  
Seasonal order `m=3` (quarterly). Supports exogenous regressors → same regressor ablation as Prophet notebook.  
**Tuned params:** `max_p`, `max_q`, `max_P`, `max_Q`; order auto-selected by AIC within those bounds.

In [ ]:
sarima_space = {
    'max_p' : hp.quniform('sarima_max_p', 1, 4, 1),
    'max_q' : hp.quniform('sarima_max_q', 1, 4, 1),
    'max_P' : hp.quniform('sarima_max_P', 0, 2, 1),
    'max_Q' : hp.quniform('sarima_max_Q', 0, 2, 1),
    'd'     : hp.quniform('sarima_d',     0, 1, 1),
    'D'     : hp.quniform('sarima_D',     0, 1, 1),
}


def sarima_cv_objective(params):
    series     = train_df[TARGET_METRIC].dropna()
    fold_mapes = []

    for tr_idx, val_idx in ts_cv_splits(series):
        tr  = series.iloc[tr_idx]
        val = series.iloc[val_idx].values
        try:
            model = pm.auto_arima(
                tr,
                seasonal      = True,
                m             = SEASONAL_PERIOD,
                d             = int(params['d']),
                D             = int(params['D']),
                max_p         = int(params['max_p']),
                max_q         = int(params['max_q']),
                max_P         = int(params['max_P']),
                max_Q         = int(params['max_Q']),
                information_criterion = 'aic',
                stepwise      = True,
                error_action  = 'ignore',
                suppress_warnings = True,
            )
            fc   = model.predict(n_periods=len(val_idx))
            fc   = np.maximum(fc, 0)
            mape = mean_absolute_percentage_error(val, fc) * 100
            fold_mapes.append(mape)
        except Exception:
            fold_mapes.append(200)

    return {'loss': np.mean(fold_mapes), 'status': STATUS_OK}


print("Tuning SARIMAX (auto_arima)...")
sarima_trials = Trials()
sarima_best_raw = fmin(
    fn        = sarima_cv_objective,
    space     = sarima_space,
    algo      = tpe.suggest,
    max_evals = 60,
    trials    = sarima_trials,
    rstate    = np.random.default_rng(42),
    verbose   = False,
)
best_sarima_params = space_eval(sarima_space, sarima_best_raw)
best_sarima_params = {k: int(v) for k, v in best_sarima_params.items()}

print(f"  Best CV MAPE : {min(sarima_trials.losses()):.2f}%")
print(f"  Target MAPE : {STATISTICAL_TARGET}%")
print("  Best params  :", best_sarima_params)

# ── Fit final model on full training data & inspect order ─────────────────
sarima_final = pm.auto_arima(
    train_df[TARGET_METRIC].dropna(),
    seasonal=True, m=SEASONAL_PERIOD,
    d=best_sarima_params['d'],   D=best_sarima_params['D'],
    max_p=best_sarima_params['max_p'],  max_q=best_sarima_params['max_q'],
    max_P=best_sarima_params['max_P'],  max_Q=best_sarima_params['max_Q'],
    information_criterion='aic', stepwise=True,
    error_action='ignore', suppress_warnings=True,
)
print(f"\n  Selected SARIMA order  : {sarima_final.order}")
print(f"  Selected seasonal order: {sarima_final.seasonal_order}")

## SARIMAX — Regressor Ablation

Same greedy forward-selection as the Prophet notebook.  
CV is done entirely within training data. Test-period exog values use `'2025 0+12'` reforecast → no leakage.

In [ ]:
candidate_regressors = [m for m in train_df.columns if m != TARGET_METRIC]
print(f"Candidate regressors: {candidate_regressors}\n")


def sarima_cv_mape_with_regressors(reg_list):
    series     = train_df[TARGET_METRIC].dropna()
    fold_mapes = []

    for tr_idx, val_idx in ts_cv_splits(series):
        tr  = series.iloc[tr_idx]
        val = series.iloc[val_idx].values

        exog_tr  = train_df[reg_list].iloc[tr_idx].values  if reg_list else None
        exog_val = train_df[reg_list].iloc[val_idx].values if reg_list else None

        # skip if any NaN in exog
        if reg_list and (
            np.any(np.isnan(exog_tr)) or np.any(np.isnan(exog_val))):
            return 200.0
        try:
            model = pm.auto_arima(
                tr, X=exog_tr,
                seasonal=True, m=SEASONAL_PERIOD,
                d=best_sarima_params['d'],   D=best_sarima_params['D'],
                max_p=best_sarima_params['max_p'],  max_q=best_sarima_params['max_q'],
                max_P=best_sarima_params['max_P'],  max_Q=best_sarima_params['max_Q'],
                information_criterion='aic', stepwise=True,
                error_action='ignore', suppress_warnings=True,
            )
            fc   = model.predict(n_periods=len(val_idx), X=exog_val)
            fc   = np.maximum(fc, 0)
            mape = mean_absolute_percentage_error(val, fc) * 100
            fold_mapes.append(mape)
        except Exception:
            fold_mapes.append(200)

    return np.mean(fold_mapes)


# ── Baseline (no regressors) ──────────────────────────────────────────────
sarima_base_mape = sarima_cv_mape_with_regressors([])
print(f"{'No regressors':<45} → CV MAPE: {sarima_base_mape:.2f}%  (baseline)\n")

# ── Single-regressor sweep ────────────────────────────────────────────────
sarima_reg_scores = {}
for reg in candidate_regressors:
    if train_df[reg].isna().any():
        print(f"  [SKIP] {reg:<38} → has NaN values")
        continue
    score = sarima_cv_mape_with_regressors([reg])
    sarima_reg_scores[reg] = score
    delta  = sarima_base_mape - score
    marker = " ✓ BETTER" if delta > 0 else ""
    print(f"  {reg:<43} → CV MAPE: {score:.2f}%  (Δ {delta:+.2f}%){marker}")

# ── Select improving regressors ───────────────────────────────────────────
sarima_improving = sorted(
    {r: s for r, s in sarima_reg_scores.items() if s < sarima_base_mape}.items(),
    key=lambda x: x[1]
)

if sarima_improving:
    sarima_best_regressors = [sarima_improving[0][0]]
    print(f"\nBest SARIMA regressor: '{sarima_best_regressors[0]}'  "
          f"CV MAPE: {sarima_improving[0][1]:.2f}%")
    # greedy forward add of 2nd regressor
    for r2, _ in sarima_improving[1:]:
        combo = sarima_cv_mape_with_regressors(sarima_best_regressors + [r2])
        print(f"  Adding '{r2}' → CV MAPE: {combo:.2f}%")
        if combo < sarima_improving[0][1]:
            sarima_best_regressors.append(r2)
            print(f"  → Added!")
            break
else:
    sarima_best_regressors = []
    print("\nNo regressor improved SARIMA CV MAPE — univariate.")

print(f"\nFinal SARIMA regressors: {sarima_best_regressors}")

# Model 4 — Theta Model

**Why:** Consistently outperforms ARIMA on short, noisy monthly series (M3/M4 competition winner).  
Decomposes series into trend (θ=0 line) and seasonality-adjusted component.  
**Tuned params:** `theta` ∈ [0.5, 3.0], `deseasonalize` (True/False), `period` ∈ {3, 6}

In [ ]:
theta_space = {
    'theta'         : hp.uniform('theta',      0.5, 3.0),
    'deseasonalize' : hp.choice('theta_deseas', [True, False]),
    'period'        : hp.choice('theta_period', [3, 6]),
}


def theta_cv_objective(params):
    series     = train_df[TARGET_METRIC].dropna()
    fold_mapes = []

    for tr_idx, val_idx in ts_cv_splits(series):
        tr  = series.iloc[tr_idx]
        val = series.iloc[val_idx].values
        try:
            model = ThetaModel(
                endog         = tr,
                period        = params['period'],
                deseasonalize = params['deseasonalize'],
            )
            fit  = model.fit(use_mle=False,   # use_mle=False → closed-form, stable on small samples
                             disp=False)
            fc   = fit.forecast(len(val_idx), theta=params['theta'])
            fc   = np.maximum(np.array(fc), 0)
            mape = mean_absolute_percentage_error(val, fc) * 100
            fold_mapes.append(mape)
        except Exception:
            fold_mapes.append(200)

    return {'loss': np.mean(fold_mapes), 'status': STATUS_OK}


print("Tuning Theta model...")
theta_trials = Trials()
theta_best_raw = fmin(
    fn        = theta_cv_objective,
    space     = theta_space,
    algo      = tpe.suggest,
    max_evals = 80,
    trials    = theta_trials,
    rstate    = np.random.default_rng(42),
    verbose   = False,
)
best_theta_params = space_eval(theta_space, theta_best_raw)

print(f"  Best CV MAPE : {min(theta_trials.losses()):.2f}%")
print(f"  Target MAPE : {STATISTICAL_TARGET}%")
print("  Best params  :", {k: (round(v, 4) if isinstance(v, float) else v)
                           for k, v in best_theta_params.items()})

# Model 5 — TBATS (Box-Cox + ARMA errors + Trigonometric Seasonality)

**Why:** Handles multiple seasonal periods simultaneously via trigonometric representation;  
Box-Cox transform stabilises variance (volatility); ARMA errors capture autocorrelated residuals.  
TBATS internally auto-selects component configuration — we grid-search over `seasonal_periods`  
and `use_box_cox` as the outer layer.  
**Grid:** `seasonal_periods` ∈ {[3], [3,6], None} × `use_box_cox` ∈ {True, False, None=auto}

In [ ]:
from itertools import product as iproduct

tbats_grid = [
    {'seasonal_periods': sp, 'use_box_cox': bc}
    for sp, bc in iproduct([[3], [3, 6], None], [True, False, None])
]

series = train_df[TARGET_METRIC].dropna()


def tbats_cv_mape(config):
    fold_mapes = []
    for tr_idx, val_idx in ts_cv_splits(series):
        tr  = series.iloc[tr_idx].values
        val = series.iloc[val_idx].values
        try:
            estimator = TBATSEstimator(
                seasonal_periods = config['seasonal_periods'],
                use_box_cox      = config['use_box_cox'],
                use_trend        = True,
                use_damped_trend = True,
                n_jobs           = 1,
            )
            fitted = estimator.fit(tr)
            fc, _  = fitted.forecast(steps=len(val_idx), confidence_level=0.95)
            fc     = np.maximum(fc, 0)
            mape   = mean_absolute_percentage_error(val, fc) * 100
            fold_mapes.append(mape)
        except Exception:
            fold_mapes.append(200)
    return np.mean(fold_mapes)


print("Grid-searching TBATS configurations...\n")
tbats_results = []
for cfg in tbats_grid:
    score = tbats_cv_mape(cfg)
    tbats_results.append((cfg, score))
    print(f"  seasonal={str(cfg['seasonal_periods']):<10}  "
          f"box_cox={str(cfg['use_box_cox']):<6}  → CV MAPE: {score:.2f}%")

best_tbats_config, best_tbats_mape = min(tbats_results, key=lambda x: x[1])
print(f"\n  Best config  : {best_tbats_config}")
print(f"  Best CV MAPE : {best_tbats_mape:.2f}%")
print(f"  Target MAPE  : {STATISTICAL_TARGET}%")

# CV Summary — All 5 Models

In [ ]:
cv_summary = pd.DataFrame([
    {'Model': 'Holt-Winters (tuned)',     'CV MAPE %': round(min(hw_trials.losses()),     2)},
    {'Model': 'ETS (AIC-selected)',        'CV MAPE %': round(min(ets_trials.losses()),    2)},
    {'Model': 'SARIMAX (auto_arima)',      'CV MAPE %': round(min(sarima_trials.losses()), 2)},
    {'Model': 'Theta',                     'CV MAPE %': round(min(theta_trials.losses()),  2)},
    {'Model': 'TBATS',                     'CV MAPE %': round(best_tbats_mape,             2)},
    {'Model': '── Statistical target ──',  'CV MAPE %': STATISTICAL_TARGET},
    {'Model': '── Baseline Prophet ──',    'CV MAPE %': BASELINE_PROPHET_MAPE},
]).set_index('Model')

print(cv_summary.to_string())

# Final Evaluation — All Models vs Actuals on Test Set

In [ ]:
train_series = train_df[TARGET_METRIC].dropna()
test_dates   = list(test_df.index)
n_test       = len(test_df)
actual_vals  = test_df[TARGET_METRIC].values
train_max    = train_series.max()
train_min    = train_series.min()

all_predictions = {}   # model_label → np.array of predictions

# ── Model 1: Holt-Winters ─────────────────────────────────────────────────
try:
    hw_model = ExponentialSmoothing(
        train_series.values,
        trend            = best_hw_params['trend'],
        seasonal         = best_hw_params['seasonal'],
        seasonal_periods = SEASONAL_PERIOD,
        damped_trend     = best_hw_params['damped_trend'],
    )
    hw_fit  = hw_model.fit(
        smoothing_level    = best_hw_params['alpha'],
        smoothing_trend    = best_hw_params['beta'],
        smoothing_seasonal = best_hw_params['gamma'],
        damping_trend      = best_hw_params['phi'],
        optimized          = False,
    )
    all_predictions['HW'] = np.maximum(hw_fit.forecast(n_test), 0)
    print(f"Holt-Winters : fitted OK")
except Exception as e:
    print(f"Holt-Winters : FAILED ({e})")

# ── Model 2: ETS ──────────────────────────────────────────────────────────
try:
    damped = best_ets_params['damped_trend'] \
             if best_ets_params['trend'] is not None else False
    ets_model = ETSModel(
        train_series.values,
        error            = best_ets_params['error'],
        trend            = best_ets_params['trend'],
        seasonal         = best_ets_params['seasonal'],
        seasonal_periods = SEASONAL_PERIOD if best_ets_params['seasonal'] else None,
        damped_trend     = damped,
    )
    ets_fit = ets_model.fit(disp=False)
    all_predictions['ETS'] = np.maximum(ets_fit.forecast(n_test), 0)
    print(f"ETS          : fitted OK")
except Exception as e:
    print(f"ETS          : FAILED ({e})")

# ── Model 3: SARIMAX ──────────────────────────────────────────────────────
try:
    exog_tr  = train_df[sarima_best_regressors].values \
               if sarima_best_regressors else None
    # test-period exog: pull '2025 0+12' reforecast values from df
    if sarima_best_regressors:
        exog_test_parts = []
        for reg in sarima_best_regressors:
            reg_test = df[
                (df['PORTFOLIO']    == portfolio) &
                (df['METRIC']       == reg) &
                (df['FORECAST_TYPE'] == '2025 0+12')
            ][['DATE', 'METRIC_VALUE']].copy()
            reg_test['DATE'] = pd.to_datetime(reg_test['DATE']).dt.date
            reg_test = (reg_test.set_index('DATE')
                        .reindex(test_dates)
                        .rename(columns={'METRIC_VALUE': reg}))
            exog_test_parts.append(reg_test)
        exog_test = pd.concat(exog_test_parts, axis=1).values
    else:
        exog_test = None

    sarima_final2 = pm.auto_arima(
        train_series, X=exog_tr,
        seasonal=True, m=SEASONAL_PERIOD,
        d=best_sarima_params['d'],   D=best_sarima_params['D'],
        max_p=best_sarima_params['max_p'],  max_q=best_sarima_params['max_q'],
        max_P=best_sarima_params['max_P'],  max_Q=best_sarima_params['max_Q'],
        information_criterion='aic', stepwise=True,
        error_action='ignore', suppress_warnings=True,
    )
    all_predictions['SARIMA'] = np.maximum(
        sarima_final2.predict(n_periods=n_test, X=exog_test), 0)
    print(f"SARIMAX      : fitted OK  order={sarima_final2.order}  "
          f"seasonal={sarima_final2.seasonal_order}")
except Exception as e:
    print(f"SARIMAX      : FAILED ({e})")

# ── Model 4: Theta ────────────────────────────────────────────────────────
try:
    theta_model = ThetaModel(
        endog         = train_series,
        period        = best_theta_params['period'],
        deseasonalize = best_theta_params['deseasonalize'],
    )
    theta_fit = theta_model.fit(use_mle=False, disp=False)
    fc_theta  = theta_fit.forecast(n_test, theta=best_theta_params['theta'])
    all_predictions['Theta'] = np.maximum(np.array(fc_theta), 0)
    print(f"Theta        : fitted OK  theta={best_theta_params['theta']:.3f}")
except Exception as e:
    print(f"Theta        : FAILED ({e})")

# ── Model 5: TBATS ────────────────────────────────────────────────────────
try:
    tbats_estimator = TBATSEstimator(
        seasonal_periods = best_tbats_config['seasonal_periods'],
        use_box_cox      = best_tbats_config['use_box_cox'],
        use_trend        = True,
        use_damped_trend = True,
        n_jobs           = 1,
    )
    tbats_fitted = tbats_estimator.fit(train_series.values)
    fc_tbats, _  = tbats_fitted.forecast(steps=n_test, confidence_level=0.95)
    all_predictions['TBATS'] = np.maximum(fc_tbats, 0)
    print(f"TBATS        : fitted OK  config={best_tbats_config}")
except Exception as e:
    print(f"TBATS        : FAILED ({e})")

print("\nAll models evaluated.")

# MAPE Comparison Table + Line Chart

In [ ]:
# ── Build long-format pred_df with every model + actuals ─────────────────
label_map = {
    'HW'    : 'HW_PRED',
    'ETS'   : 'ETS_PRED',
    'SARIMA': 'SARIMA_PRED',
    'Theta' : 'THETA_PRED',
    'TBATS' : 'TBATS_PRED',
}

rows = [{'DATE': d, 'FORECAST_TYPE': 'Actual', 'METRIC_VALUE': a}
        for d, a in zip(test_dates, actual_vals)]

for short, label in label_map.items():
    if short in all_predictions:
        for d, p in zip(test_dates, all_predictions[short]):
            rows.append({'DATE': d, 'FORECAST_TYPE': label, 'METRIC_VALUE': p})

pred_df_all = pd.DataFrame(rows)

# ── Attach Original and 2025 0+12 from Snowflake df ──────────────────────
pred_df_all = add_reforecast_data(
    pred_df        = pred_df_all,
    main_df        = df,
    portfolio      = portfolio,
    metric         = TARGET_METRIC,
    forecast_types = ['Original', '2025 0+12'],
)
pred_df_all['DATE'] = pred_df_all['DATE'].apply(
    lambda x: x.date() if isinstance(x, pd.Timestamp) else x)

# ── MAPE table ────────────────────────────────────────────────────────────
y_true    = (pred_df_all[pred_df_all['FORECAST_TYPE'] == 'Actual']
             .sort_values('DATE').set_index('DATE')['METRIC_VALUE'])
result_df = pd.DataFrame({'Actual': y_true.values}, index=y_true.index)

all_cols = ['Original', '2025 0+12'] + list(label_map.values())
for col in all_cols:
    y_pred = (pred_df_all[pred_df_all['FORECAST_TYPE'] == col]
              .sort_values('DATE').set_index('DATE')['METRIC_VALUE'])
    y_pred.name = col
    result_df = result_df.merge(y_pred, how='left',
                                left_index=True, right_index=True)

print(f"\n{'='*60}")
print(f"MAPE Comparison — {TARGET_METRIC}")
print(f"{'='*60}")
mape_rows = []
for col in result_df.columns:
    if col == 'Actual':
        continue
    mask = result_df[col].notna()
    if mask.sum() == 0:
        continue
    mape = mean_absolute_percentage_error(
        y_true=result_df['Actual'][mask],
        y_pred=result_df[col][mask]) * 100
    tag = "  ← BEST" if mape == min([
        mean_absolute_percentage_error(
            result_df['Actual'][result_df[c].notna()],
            result_df[c][result_df[c].notna()]) * 100
        for c in result_df.columns if c != 'Actual' and result_df[c].notna().sum() > 0
    ]) else ""
    beats = " ✓ beats target" if mape < STATISTICAL_TARGET else ""
    print(f"  {col:<22} {mape:6.2f}%{tag}{beats}")
    mape_rows.append({'Model': col, 'Test MAPE %': round(mape, 2)})

print(f"  {'─'*40}")
print(f"  Statistical target    {STATISTICAL_TARGET:6.2f}%")
print(f"  Baseline Prophet      {BASELINE_PROPHET_MAPE:6.2f}%")

# ── Line chart ────────────────────────────────────────────────────────────
plot_results(pred_df_all, TARGET_METRIC)
plt.show()

# Diagnostics — Best Model Residual Analysis

ACF/PACF of residuals for the best-performing model.  
**Goal:** residuals should resemble white noise — no significant spikes beyond lag 0 → model has captured all structure.

In [ ]:
# Identify the best model on test MAPE
best_test_mape = 999
best_model_label = None
for col in result_df.columns:
    if col == 'Actual':
        continue
    mask = result_df[col].notna()
    if mask.sum() == 0:
        continue
    mape = mean_absolute_percentage_error(
        result_df['Actual'][mask], result_df[col][mask]) * 100
    if mape < best_test_mape:
        best_test_mape  = mape
        best_model_label = col

print(f"Best model on test set: {best_model_label} ({best_test_mape:.2f}%)\n")

# ── Re-fit best model on training data to extract in-sample residuals ─────
reverse_label = {v: k for k, v in label_map.items()}
best_short = reverse_label.get(best_model_label, None)

residuals = None

if best_short == 'HW':
    hw_diag = ExponentialSmoothing(
        train_series.values,
        trend=best_hw_params['trend'], seasonal=best_hw_params['seasonal'],
        seasonal_periods=SEASONAL_PERIOD,
        damped_trend=best_hw_params['damped_trend'],
    ).fit(smoothing_level=best_hw_params['alpha'],
          smoothing_trend=best_hw_params['beta'],
          smoothing_seasonal=best_hw_params['gamma'],
          damping_trend=best_hw_params['phi'], optimized=False)
    residuals = hw_diag.resid

elif best_short == 'ETS':
    damped = best_ets_params['damped_trend'] \
             if best_ets_params['trend'] is not None else False
    ets_diag = ETSModel(
        train_series.values, error=best_ets_params['error'],
        trend=best_ets_params['trend'], seasonal=best_ets_params['seasonal'],
        seasonal_periods=SEASONAL_PERIOD if best_ets_params['seasonal'] else None,
        damped_trend=damped,
    ).fit(disp=False)
    residuals = ets_diag.resid

elif best_short == 'SARIMA':
    residuals = sarima_final2.resid()

elif best_short == 'Theta':
    theta_diag = ThetaModel(
        train_series,
        period=best_theta_params['period'],
        deseasonalize=best_theta_params['deseasonalize'],
    ).fit(use_mle=False, disp=False)
    residuals = theta_diag.resid

elif best_short == 'TBATS':
    tbats_diag = TBATSEstimator(
        seasonal_periods=best_tbats_config['seasonal_periods'],
        use_box_cox=best_tbats_config['use_box_cox'],
        use_trend=True, use_damped_trend=True, n_jobs=1,
    ).fit(train_series.values)
    residuals = tbats_diag.resid

if residuals is not None:
    residuals = np.array(residuals).flatten()
    fig, axes = plt.subplots(2, 2, figsize=(18, 8))
    fig.suptitle(f'Residual Diagnostics — {best_model_label}', fontsize=14)

    # Time-series of residuals
    axes[0, 0].plot(residuals, marker='o')
    axes[0, 0].axhline(0, color='r', linestyle='--')
    axes[0, 0].set_title('Residuals over time')
    axes[0, 0].set_xlabel('Time index')
    axes[0, 0].set_ylabel('Residual')

    # Histogram
    axes[0, 1].hist(residuals, bins=10, edgecolor='black')
    axes[0, 1].set_title('Residual Distribution')
    axes[0, 1].set_xlabel('Residual')

    # ACF
    plot_acf(residuals, ax=axes[1, 0], lags=min(10, len(residuals) - 2),
             title='ACF of Residuals')

    # PACF
    plot_pacf(residuals, ax=axes[1, 1], lags=min(10, len(residuals) // 2 - 1),
              title='PACF of Residuals', method='ywm')

    plt.tight_layout()
    plt.show()
    print("\nInterpretation: ACF/PACF spikes within blue bands → white noise ✓")
    print("  Spikes outside bands → remaining autocorrelation (model under-specified)")
else:
    print("Residuals not available for this model type.")

# Final Summary

In [ ]:
summary_rows = []
for col in result_df.columns:
    if col == 'Actual':
        continue
    mask = result_df[col].notna()
    if mask.sum() == 0:
        continue
    test_mape = mean_absolute_percentage_error(
        result_df['Actual'][mask], result_df[col][mask]) * 100
    col_short = col.replace('_PRED', '')
    cv_m = {
        'HW_PRED'     : round(min(hw_trials.losses()),      2),
        'ETS_PRED'    : round(min(ets_trials.losses()),     2),
        'SARIMA_PRED' : round(min(sarima_trials.losses()),  2),
        'THETA_PRED'  : round(min(theta_trials.losses()),   2),
        'TBATS_PRED'  : round(best_tbats_mape,              2),
    }.get(col, '—')
    beats = '✓' if test_mape < STATISTICAL_TARGET else ''
    summary_rows.append({
        'Model'         : col,
        'CV MAPE %'     : cv_m,
        'Test MAPE %'   : round(test_mape, 2),
        'Beats 3.21%'   : beats,
    })

summary_rows += [
    {'Model': '── Statistical baseline ──', 'CV MAPE %': '—', 'Test MAPE %': STATISTICAL_TARGET, 'Beats 3.21%': ''},
    {'Model': '── Prophet baseline ──',     'CV MAPE %': '—', 'Test MAPE %': BASELINE_PROPHET_MAPE, 'Beats 3.21%': ''},
]

summary_df = pd.DataFrame(summary_rows).set_index('Model')
print(summary_df.to_string())